<a href="https://colab.research.google.com/github/fcoliveira-utfpr/aquacrop_ml/blob/main/treinamento_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Início - Bibliotecas**
---

In [1]:
# Para acessar os dados, basta executar este bloco de código
!pip install xgboost optuna    -q
!pip install lightgbm    -q
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score
from sklearn.neural_network import MLPRegressor
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn import linear_model
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 5.6 MB/s eta 0:00:00


In [4]:
url = "https://raw.githubusercontent.com/fcoliveira-utfpr/aquacrop_ml/refs/heads/main/df_wide.csv"
df_wide = pd.read_csv(url)
#df_wide

In [9]:
corr = df_wide.corr(numeric_only=True)['Yield_obs'].drop('Yield_obs')
top20 = corr.abs().sort_values(ascending=False).head(20)
top20

,Yield_obs
UR_F1,0.256040
ISNA_F3,0.240807
Amp_F3,0.232867
Tmax_F1,0.225948
ETR_F3,0.217399
ARM_F3,0.214009
Tmed_F1,0.209612
Amp_F4,0.203957
Chuva_F3,0.197827
T_DEF_F1,0.191169


#RandomForest

In [ ]:
# ==========================================================
# RANDOM FOREST + OPTUNA
# ==========================================================

import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==========================================================
# ORGANIZAR DATAFRAME
# ==========================================================

# FEATURES
X = df_wide[top20.index]

# TARGET
y = df_wide['Yield_obs']

# ==========================================================
# TRAIN / TEST SPLIT TEMPORAL
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)

# ==========================================================
# VALIDAÇÃO CRUZADA
# ==========================================================

from sklearn.model_selection import KFold

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ==========================================================
# FUNÇÃO OBJETIVO DO OPTUNA
# ==========================================================

def objective(trial):

    params = {

        "n_estimators": trial.suggest_int("n_estimators", 200, 600),

        "max_depth": trial.suggest_int("max_depth", 3, 30),

        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),

        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),

        "max_features": trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2", None]
        ),

        "bootstrap": trial.suggest_categorical(
            "bootstrap",
            [True, False]
        )
    }

    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    # converter erro negativo para positivo
    return -scores.mean()

# ==========================================================
# CRIAR ESTUDO OPTUNA
# ==========================================================

study = optuna.create_study(direction="minimize")

print("\n🔎 Iniciando otimização com Optuna...\n")

study.optimize(
    objective,
    n_trials=40,
    show_progress_bar=True
)

# ==========================================================
# MELHORES HIPERPARÂMETROS
# ==========================================================

print("\n🏆 Melhores hiperparâmetros encontrados:")
print(study.best_params)

# ==========================================================
# TREINAR MODELO FINAL
# ==========================================================

best_model = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)

# ==========================================================
# PREVISÃO
# ==========================================================

y_pred = best_model.predict(X_test)

# ==========================================================
# MÉTRICAS
# ==========================================================

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("\n" + "="*60)
print("RESULTADOS NO CONJUNTO DE TESTE")
print("="*60)

print(f"R²   : {r2:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")

# ==========================================================
# IMPORTÂNCIA DAS VARIÁVEIS
# ==========================================================

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n" + "="*60)
print("IMPORTÂNCIA DAS FEATURES")
print("="*60)

print(importance)

# ==========================================================
# GRÁFICO DE IMPORTÂNCIA
# ==========================================================

import matplotlib.pyplot as plt

importance.plot(
    x='Feature',
    y='Importance',
    kind='bar',
    figsize=(10,5),
    legend=False
)

plt.title('Feature Importance - Random Forest')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

[I 2026-03-16 03:41:48,362] A new study created in memory with name: no-name-aa51bc37-e634-497c-b25f-9cda8a1cd40a



🔎 Iniciando otimização com Optuna...



  0%|          | 0/40 [00:00<?, ?it/s]

#XGBRegressor

In [ ]:
# ==========================================================
# OTIMIZAÇÃO DE XGBOOST COM OPTUNA
# Partindo de X (DataFrame) e y (Series)
# ==========================================================

import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor


# ==========================================================
# GARANTIR ORDEM TEMPORAL (importante para séries)
# ==========================================================

X = X.sort_index()
y = y.loc[X.index]


# ==========================================================
# TRAIN / TEST SPLIT TEMPORAL
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)
# Extrair Yield_obs do X_train e X_test
yield_obs_train = X_train['Yield_obs']
yield_obs_test = X_test['Yield_obs'].values
X_train = X_train.drop('Yield_obs', axis=1)
X_test = X_test.drop('Yield_obs', axis=1)

# ==========================================================
# VALIDAÇÃO CRUZADA TEMPORAL
# ==========================================================

tscv = TimeSeriesSplit(n_splits=5)


# ==========================================================
# FUNÇÃO OBJETIVO DO OPTUNA
# ==========================================================

def objective(trial):

    params = {

        "n_estimators": trial.suggest_int("n_estimators", 100, 800),

        "max_depth": trial.suggest_int("max_depth", 3, 12),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.5,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.5,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-8,
            10,
            log=True
        )
    }

    model = XGBRegressor(
        **params,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=tscv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    return scores.mean()


# ==========================================================
# CRIAR ESTUDO OPTUNA
# ==========================================================

study = optuna.create_study(
    direction="maximize"
)

print("\n🔎 Iniciando otimização com Optuna (XGBoost)...\n")

study.optimize(
    objective,
    n_trials=40,
    show_progress_bar=True
)


# ==========================================================
# MELHORES HIPERPARÂMETROS
# ==========================================================

print("\n🏆 Melhores hiperparâmetros encontrados:")
print(study.best_params)


# ==========================================================
# TREINAR MODELO FINAL
# ==========================================================

best_model = XGBRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

best_model.fit(X_train, y_train)


# ==========================================================
# PREVISÃO
# ==========================================================

y_pred = best_model.predict(X_test)


# ==========================================================
# MÉTRICAS
# ==========================================================

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100


print("\n" + "="*60)
print("RESULTADOS NO CONJUNTO DE TESTE")
print("="*60)

print(f"R²   : {r2:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")

[I 2026-03-14 18:54:16,074] A new study created in memory with name: no-name-55a7e23b-4fc4-4b37-9caa-9a7ce9e03aeb



🔎 Iniciando otimização com Optuna (XGBoost)...



  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-03-14 18:54:49,011] Trial 0 finished with value: -0.9145007290017977 and parameters: {'n_estimators': 720, 'max_depth': 7, 'learning_rate': 0.14334730040579943, 'subsample': 0.664288814312042, 'colsample_bytree': 0.9292508614797967, 'gamma': 4.141287127617156, 'reg_alpha': 0.012276694773143103, 'reg_lambda': 1.971706414639806e-05}. Best is trial 0 with value: -0.9145007290017977.
[I 2026-03-14 18:55:15,791] Trial 1 finished with value: -0.9226769569613118 and parameters: {'n_estimators': 399, 'max_depth': 3, 'learning_rate': 0.019587849472978775, 'subsample': 0.7123174386943387, 'colsample_bytree': 0.5695624626013791, 'gamma': 0.46051222712197415, 'reg_alpha': 1.2276174829748788e-07, 'reg_lambda': 0.7174859499205423}. Best is trial 0 with value: -0.9145007290017977.
[I 2026-03-14 18:55:43,015] Trial 2 finished with value: -0.9162547672766586 and parameters: {'n_estimators': 503, 'max_depth': 12, 'learning_rate': 0.23325117836900405, 'subsample': 0.6024766276175555, 'colsample_b

#ExtraTreesRegressor

In [ ]:
# ==========================================================
# OTIMIZAÇÃO DE EXTRA TREES COM OPTUNA
# Partindo de X (DataFrame) e y (Series)
# ==========================================================

import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ==========================================================
# GARANTIR ORDEM TEMPORAL
# ==========================================================

X = X.sort_index()
y = y.loc[X.index]


# ==========================================================
# TRAIN / TEST SPLIT
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)
# Extrair Yield_obs do X_train e X_test
yield_obs_train = X_train['Yield_obs']
yield_obs_test = X_test['Yield_obs'].values
X_train = X_train.drop('Yield_obs', axis=1)
X_test = X_test.drop('Yield_obs', axis=1)

# ==========================================================
# VALIDAÇÃO TEMPORAL
# ==========================================================

tscv = TimeSeriesSplit(n_splits=5)


# ==========================================================
# FUNÇÃO OBJETIVO
# ==========================================================

def objective(trial):

    params = {

        "n_estimators": trial.suggest_int("n_estimators", 100, 800),

        "max_depth": trial.suggest_int("max_depth", 3, 30),

        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),

        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),

        "max_features": trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2", None]
        ),

        "bootstrap": trial.suggest_categorical(
            "bootstrap",
            [True, False]
        )
    }

    model = ExtraTreesRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=tscv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    return scores.mean()


# ==========================================================
# CRIAR ESTUDO
# ==========================================================

study = optuna.create_study(direction="maximize")

print("\n🔎 Iniciando otimização com Optuna (ExtraTrees)...\n")

study.optimize(
    objective,
    n_trials=40,
    show_progress_bar=True
)


# ==========================================================
# MELHORES PARÂMETROS
# ==========================================================

print("\n🏆 Melhores hiperparâmetros encontrados:")
print(study.best_params)


# ==========================================================
# MODELO FINAL
# ==========================================================

best_model = ExtraTreesRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)


# ==========================================================
# PREVISÃO
# ==========================================================

y_pred = best_model.predict(X_test)


# ==========================================================
# MÉTRICAS
# ==========================================================

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100


print("\n" + "="*60)
print("RESULTADOS NO CONJUNTO DE TESTE")
print("="*60)

print(f"R²   : {r2:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")

[I 2026-03-14 19:15:19,255] A new study created in memory with name: no-name-6ca8fe15-1816-4aa9-9296-9cf42e00ea85



🔎 Iniciando otimização com Optuna (ExtraTrees)...



  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-03-14 19:19:14,654] Trial 0 finished with value: -0.9148249650642158 and parameters: {'n_estimators': 286, 'max_depth': 25, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': None, 'bootstrap': True}. Best is trial 0 with value: -0.9148249650642158.
[I 2026-03-14 19:20:24,695] Trial 1 finished with value: -0.986398858318742 and parameters: {'n_estimators': 421, 'max_depth': 8, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: -0.9148249650642158.
[I 2026-03-14 19:22:03,863] Trial 2 finished with value: -0.9868799534977821 and parameters: {'n_estimators': 628, 'max_depth': 8, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 0 with value: -0.9148249650642158.


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[I 2026-03-14 19:26:43,197] Trial 3 finished with value: -0.9140604296037542 and parameters: {'n_estimators': 799, 'max_depth': 28, 'min_samples_split': 3, 'min_samples_leaf': 10, 'max_features': 'log2', 'bootstrap': False}. Best is trial 3 with value: -0.9140604296037542.
[I 2026-03-14 19:28:11,995] Trial 4 finished with value: -0.9644320971972148 and parameters: {'n_estimators': 505, 'max_depth': 9, 'min_samples_split': 8, 'min_samples_leaf': 6, 'max_features': 'log2', 'bootstrap': True}. Best is trial 3 with value: -0.9140604296037542.
[I 2026-03-14 19:29:41,591] Trial 5 finished with value: -0.9279961415378072 and parameters: {'n_estimators': 410, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 'sqrt', 'bootstrap': True}. Best is trial 3 with value: -0.9140604296037542.
[I 2026-03-14 19:30:50,029] Trial 6 finished with value: -1.0124099773641784 and parameters: {'n_estimators': 470, 'max_depth': 7, 'min_samples_split': 20, 'min_samples_leaf': 2, 'max

#LGBMRegressor

In [ ]:
# ==========================================================
# OTIMIZAÇÃO DE LIGHTGBM COM OPTUNA
# Partindo de X (DataFrame) e y (Series)
# ==========================================================

import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb

from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ==========================================================
# GARANTIR ORDEM TEMPORAL
# ==========================================================

X = X.sort_index()
y = y.loc[X.index]


# ==========================================================
# TRAIN / TEST SPLIT TEMPORAL
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)
# Extrair Yield_obs do X_train e X_test
yield_obs_train = X_train['Yield_obs']
yield_obs_test = X_test['Yield_obs'].values
X_train = X_train.drop('Yield_obs', axis=1)
X_test = X_test.drop('Yield_obs', axis=1)

# ==========================================================
# VALIDAÇÃO TEMPORAL
# ==========================================================

tscv = TimeSeriesSplit(n_splits=5)


# ==========================================================
# FUNÇÃO OBJETIVO
# ==========================================================

def objective(trial):

    params = {

        "n_estimators": trial.suggest_int("n_estimators", 100, 800),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.3,
            log=True
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            20,
            300
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            15
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            5,
            100
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.5,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.5,
            1.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-8,
            10,
            log=True
        )
    }

    model = lgb.LGBMRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=tscv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    return scores.mean()


# ==========================================================
# CRIAR ESTUDO
# ==========================================================

study = optuna.create_study(direction="maximize")

print("\n🔎 Iniciando otimização com Optuna (LightGBM)...\n")

study.optimize(
    objective,
    n_trials=40,
    show_progress_bar=True
)


# ==========================================================
# MELHORES PARÂMETROS
# ==========================================================

print("\n🏆 Melhores hiperparâmetros encontrados:")
print(study.best_params)


# ==========================================================
# TREINAR MODELO FINAL
# ==========================================================

best_model = lgb.LGBMRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)


# ==========================================================
# PREVISÃO
# ==========================================================

y_pred = best_model.predict(X_test)


# ==========================================================
# MÉTRICAS
# ==========================================================

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100


print("\n" + "="*60)
print("RESULTADOS NO CONJUNTO DE TESTE")
print("="*60)

print(f"R²   : {r2:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.2f}%")